# APIGEE Ingestion Daily — Fixed

## Root Cause
Two symptoms, one cause: **`size: 10000` without a sort** caused both the total
undercount (~5k less than the portal) AND the per-journey miscounts
(3 journeys overcounted, 2 journeys undercounted).

### Why the total was less
Peak daytime intervals have more than 10,000 records. Elasticsearch returns exactly
10,000 and silently drops the rest — no error, no warning.

### Why per-journey counts were wrong in BOTH directions
Without an explicit `sort`, Elasticsearch returns documents in **shard/segment order**,
not timestamp order. This creates a biased sample:
- Records from shards that respond first dominate the 10,000 slots
  → those journeys get **inflated** counts (3 overcounted journeys)
- Records from slower shards are cut off
  → those journeys get **deflated** counts (2 undercounted journeys)

The bias is consistent across runs because shard assignment is deterministic per index.
That is why the same 3 journeys always overcounted and the same 2 always undercounted.

## Fix
1. Added `sort: [{"@timestamp": "asc"}, {"_id": "asc"}]` — mandatory for deterministic
   ordering and correct `search_after` cursor positioning.
2. Added `search_after` pagination — loops within each interval until all records are
   fetched (batch size < 10,000), so the 10k cap is never a ceiling.

Once all records are retrieved without truncation, each journey gets its correct
proportion and both the total and per-journey counts will match the portal.

In [ ]:
import requests
import pytz
from datetime import datetime, timedelta

kolkata_tz = pytz.timezone("Asia/Kolkata")

user = "apigee_dbp"
psd  = "apigee@123"
url  = "https://10.227.12.188:9201/apigee_updated_solace/_search"

headers = {
    "Content-Type": "application/vnd.elasticsearch+json; compatible-with=8",
    "Accept":       "application/vnd.elasticsearch+json; compatible-with=8",
}

# Replace with your actual SCOPE_JOURNEY_MAP
SCOPE_JOURNEY_MAP = {
    "TDCC": [],
    "SNCC": [],
    "PPCC": [],
    "PTCC": [],
}
all_scopes = list(SCOPE_JOURNEY_MAP.keys())

file_date       = (datetime.now(kolkata_tz).date() - timedelta(days=1)).strftime("%Y-%m-%d")
interval_minute = 30                              # changed from 10 → 30 mins (48 intervals/day)
total_intervals = (24 * 60) // interval_minute   # 48

print(f"file_date      : {file_date}")
print(f"total_intervals: {total_intervals}")

In [ ]:
all_hits = []

for i in range(total_intervals):
    start_total_min = i * interval_minute
    start_hour      = start_total_min // 60
    start_minute    = start_total_min % 60

    end_total_min = start_total_min + interval_minute - 1
    end_hour      = end_total_min // 60
    end_minute    = end_total_min % 60

    start_time = f"{file_date}T{start_hour:02d}:{start_minute:02d}:00.000"
    end_time   = f"{file_date}T{end_hour:02d}:{end_minute:02d}:59.999"
    print(f"Fetching interval {i + 1}/{total_intervals}: {start_time} to {end_time}")

    interval_hits = []
    search_after  = None
    page          = 0

    while True:
        page += 1
        query_body = {
            "size":             10000,
            "fields":           ["*"],
            "track_total_hits": True,   # shows actual count in ES vs what we fetched
            # _doc sort: fast, unique per shard, recommended tiebreaker for search_after
            # DO NOT use _id sort — it hits a low internal limit in many ES 8.x clusters
            "sort": [{"@timestamp": "asc"}, {"_doc": "asc"}],
            "query": {
                "bool": {
                    "must": [
                        {"range": {"@timestamp": {"gte": start_time, "lte": end_time}}},
                        {"terms": {"Scope.keyword": all_scopes}},
                    ]
                }
            },
        }

        if search_after:
            query_body["search_after"] = search_after

        response = requests.post(
            url,
            headers=headers,
            auth=(user, psd),
            json=query_body,
            verify=False,
            timeout=120,
        )

        if response.status_code != 200:
            raise Exception(
                f"API call failed for interval {start_time} - {end_time} "
                f"(page {page}): {response.text[-200]}"
            )

        resp_json     = response.json()
        total_in_es   = resp_json["hits"]["total"]["value"]
        hits          = resp_json.get("hits", {}).get("hits", [])
        interval_hits.extend(hits)

        # On page 1, warn if this interval will need multiple pages
        if page == 1 and total_in_es > 10000:
            print(f"  ⚠️  interval has {total_in_es} records — paginating ...")

        if len(hits) < 10000:
            # Sanity check: if we got less than size but total says more, something is wrong
            if len(interval_hits) < total_in_es:
                print(f"  ⚠️  WARNING: fetched {len(interval_hits)} but ES reports {total_in_es} — check sort/index config")
            break

        # Full page returned — advance the cursor and fetch next page
        search_after = hits[-1]["sort"]

    all_hits.extend(interval_hits)
    pages_info = f" [{page} pages]" if page > 1 else ""
    print(f"interval records: {len(interval_hits)}{pages_info} | running total: {len(all_hits)}")

print(f"\nAll {total_intervals} intervals fetched successfully. Total records: {len(all_hits)}")